Build a GraphRAG Pipeline that:

1. Reads text.
2. Extracts relationships (Subject -> Predicate -> Object).
3. Builds a Graph using NetworkX.
4. Retrieves context by walking the graph (Multi-hop reasoning).
5. Answers a question based on that deep context.

In [3]:
!pip install networkx

Step 1: Loading the LLM

LLM to do the main work, like reading text and pulling out logic

In [6]:
import networkx as nx
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Load Local LLM
# Temperature=0 is crucial here. We want facts, not creativity.
llm = ChatOllama(model="mistral", temperature=0)

Step 2: Turning Text into Data

triple: (Head) -> [Relation] -> (Tail)

In [12]:
# 2. Prompt for Extracting Graph Triples
extract_prompt = PromptTemplate(
    template="""
You are an expert knowledge graph builder.
Extract entities and relationships from the text.
Return ONLY a JSON list. Each item must contain:
- "head": source entity
- "relation": relationship
- "tail": target entity

Text:
{text}

Output JSON:
""",
    input_variables=["text"],
)

extraction_chain = extract_prompt | llm | JsonOutputParser()

Step 3: The Data Source

In [17]:
# 3. Enterprise Knowledge Example
company_text = """
OpenAI was founded by Sam Altman and Elon Musk.
OpenAI developed GPT-4.
GPT-4 powers ChatGPT.
Microsoft partnered with OpenAI.
Microsoft invested 10 billion dollars in OpenAI.
ChatGPT is used by millions of users worldwide.
"""

print("\n Extracting knowledge graph triples...\n")
triples = extraction_chain.invoke({"text": company_text})
print(triples)


 Extracting knowledge graph triples...

[{'head': 'OpenAI', 'relation': 'founded_by', 'tail': 'Sam Altman'}, {'head': 'OpenAI', 'relation': 'founded_by', 'tail': 'Elon Musk'}, {'head': 'OpenAI', 'relation': 'developed', 'tail': 'GPT-4'}, {'head': 'GPT-4', 'relation': 'powers', 'tail': 'ChatGPT'}, {'head': 'OpenAI', 'relation': 'partnered_with', 'tail': 'Microsoft'}, {'head': 'Microsoft', 'relation': 'invested_in', 'tail': 'OpenAI'}, {'head': 'Microsoft', 'relation': 'invested_amount', 'tail': '10 billion dollars'}, {'head': 'ChatGPT', 'relation': 'used_by', 'tail': 'millions of users worldwide'}]


Step 4: Building the Graph

In [20]:
# 4. Build Knowledge Graph
kg = nx.DiGraph() # DiGraph means "Directed Graph" (arrows point one way)

def build_knowledge_graph(triples):
    for item in triples:
        head = item.get("head")
        tail = item.get("tail")
        relation = item.get("relation")

        if head and tail:
            kg.add_node(head)
            kg.add_node(tail)
            kg.add_edge(head, tail, label=relation)

build_knowledge_graph(triples)

print("\n Nodes in Graph:")
print(list(kg.nodes()))


 Nodes in Graph:
['OpenAI', 'Sam Altman', 'Elon Musk', 'GPT-4', 'ChatGPT', 'Microsoft', '10 billion dollars', 'millions of users worldwide']


Step 5: Multi-Hop

With GraphRAG, we start at “ChatGPT” and explore its connections:
1. Start at ChatGPT.
2. Look backward: “Powered by GPT-4”.
3. Walk to GPT-4: “Developed by OpenAI”.
4. Walk to OpenAI: “Invested in by Microsoft”.

In [24]:
# 5. MULTI-HOP RETRIEVAL
def retrieve_graph_context(entity, max_depth=2):
    context = set()
    visited_nodes = set()

    def dfs(node, depth):
        if depth > max_depth:
            return
        visited_nodes.add(node)

        # 1. Check Outgoing edges (What does this node do?)
        for neighbor in kg.successors(node):
            relation = kg.get_edge_data(node, neighbor)["label"]
            context.add(f"{node} {relation} {neighbor}")
            if neighbor not in visited_nodes:
                dfs(neighbor, depth + 1)

        # 2. Check Incoming edges (Who interacts with this node?)
        for predecessor in kg.predecessors(node):
            relation = kg.get_edge_data(predecessor, node)["label"]
            context.add(f"{predecessor} {relation} {node}")
            if predecessor not in visited_nodes:
                dfs(predecessor, depth + 1)

    if entity in kg.nodes:
        dfs(entity, 1) # Start the traversal

    return ". ".join(context)

Step 6: The Final Answer

In [29]:
# 6. Final RAG Prompt
final_prompt = PromptTemplate(
    template="""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

rag_chain = final_prompt | llm

# 7. Ask a Multi-hop Reasoning Question
entity = "ChatGPT"

# We ask for a depth of 3 to catch distant connections
graph_context = retrieve_graph_context(entity, max_depth=3) 

print("\n Retrieved Graph Context:\n")
print(graph_context)

question = "Which company invested in the company that built ChatGPT?"

response = rag_chain.invoke({
    "context": graph_context,
    "question": question
})

print("\n Final Answer:\n")
print(response.content)


 Retrieved Graph Context:

Microsoft invested_in OpenAI. OpenAI founded_by Sam Altman. OpenAI founded_by Elon Musk. GPT-4 powers ChatGPT. OpenAI developed GPT-4. OpenAI partnered_with Microsoft. ChatGPT used_by millions of users worldwide

 Final Answer:

 Microsoft invested in the company (OpenAI) that built ChatGPT.


Uses of GraphRAG Pipeline:
- In fraud detection, GraphRAG can connect a suspicious phone number to an address, a previous claim, and a known fraudster.
- In medical research, it can connect Drug X to Protein Y to Disease Z across thousands of research papers.